# 2단계 · 연관 네트워크 확장

**목표:** 확정한 시드와 **함께 부상하는 연관 아이템들**을 APOLLO 네트워크 API로 끌어옵니다. (A4.4)

## 개념
- `/items/{id}/network?degree=N` 은 시드를 중심으로 한 **연관 네트워크**(노드 + 엣지)를 반환합니다.
- **노드** = 아이템 `{id, label, category, level}`  ·  **엣지** = 아이템 간 연관 `{from, to}` (이름 기준)
- `level` = 시드로부터의 거리(0=시드, 1=직접 이웃, 2=이웃의 이웃)

## degree(차수)의 의미 — 중요
| API degree | 결과 | 체감 차수 |
|---|---|---|
| 1 | 빈 결과 | — |
| 2 | 시드 + 직접 이웃 (별 모양, ~11노드) | 1차수 |
| **3** | 이웃의 이웃까지 (~수십~97노드, 노드 간 연결 있음) | **2차수 (권장)** |

즉 **의미 있는 네트워크 분석은 degree=3(2차수)** 부터입니다.

> **1단계에서 확정한 시드를 사용합니다.**
> 데모 시드: **Mathematical finance** (itemId `33589680`, 카테고리 *인공지능*, 지표 *기술집약도*)

In [ ]:
# --- APOLLO API 호출 헬퍼 (모든 노트북 공통) ---
import requests, urllib3
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://apollo.kisti.re.kr/service-test"   # 테스트 서버

def call_api(method, path, params=None, payload=None):
    """APOLLO 호출 후 {meta, data} 본문을 반환. (테스트 서버라 verify=False)"""
    r = requests.request(method, BASE_URL + path, params=params, json=payload,
                         headers={"Content-Type": "application/json"},
                         verify=False, timeout=120)
    body = r.json()
    if not (isinstance(body, dict) and "data" in body):
        raise RuntimeError(f"API 오류 (HTTP {r.status_code}): {body}")
    return body

print("준비 완료 · BASE_URL =", BASE_URL)

In [ ]:
# 1단계에서 확정한 시드 (다른 데모 사례로 바꾸려면 값만 교체)
SEED_ID = 33589680          # Mathematical finance
SEED_NAME = "Mathematical finance"
CATEGORY = "인공지능"
DEGREE = 3                  # APOLLO degree=3  →  사용자 체감 '2차수'(이웃의 이웃)

### 연관 네트워크 호출 (A4.4)

In [ ]:
res = call_api("GET", f"/open/api/v1/items/{SEED_ID}/network",
               params={"degree": DEGREE, "indicator": "true"})
data = res["data"]
nodes = data.get("nodes", [])
edges = data.get("edges", [])
print(f"네트워크 확장 (degree={DEGREE}) → 노드 {len(nodes)}개, 엣지 {len(edges)}개")

# 차수(level)별 노드 수
lv = pd.Series([n.get("level") for n in nodes]).value_counts().sort_index()
print("\n차수(level)별 노드 수:")
print(lv.to_string())

### 노드 / 엣지 살펴보기

In [ ]:
print("=== 노드 예시 ===")
display(pd.DataFrame(nodes).head(8))
print("=== 엣지 예시 (from → to) ===")
display(pd.DataFrame(edges).head(8))

**정리**
- degree=3 으로 시드 주변 **2차수 네트워크**를 확보했습니다.
- 이 노드들이 곧 3단계에서 지표를 수집할 **연관 아이템 후보**입니다.